# Loose coupling of FreeGSNKE with TORAX

This notebook demonstrates a **loosely coupled** simulation of the plasma core transport (with [TORAX](https://github.com/google-deepmind/torax)) and the free-boundary equilibrium (with FreeGSNKE).

In loose coupling the two codes are advanced alternately over each coupling interval $[t, t+\Delta t]$ and iterated to convergence, as opposed to tight coupling where the transport and equilibrium equations would be solved together as a single system:

1. TORAX provides the profiles that source the Grad-Shafranov equation, $p'(\psi)$ and $FF'(\psi)$, together with the plasma current $I_p$, as an IMAS `equilibrium` IDS.
2. FreeGSNKE solves the static free-boundary equilibrium for those profiles (and the coil currents at $t + \Delta t$) and returns the new geometry as an IMAS `equilibrium` IDS.
3. TORAX builds its flux-surface-averaged geometry from that IDS and takes a (jitted) transport step from $t$ to $t + \Delta t$ using the geometry at both ends of the interval.
4. Steps 1-3 are repeated (with under-relaxation) until the exchanged $p'$ and $FF'$ profiles stop changing, after which the interval is accepted.

The IMAS `equilibrium` IDS is the interchange format in both directions: see `freegsnke.imas_read_write` for how FreeGSNKE equilibria are written to (and profiles read from) IDSs, and `freegsnke.torax_coupling` for the coupling driver.

**Requirements**: TORAX must be installed (`pip install freegsnke[torax]`, see the README for details on the `imas-python` version pins).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torax
from freegsnke import build_machine, equilibrium_update, GSstaticsolver, torax_coupling

## Initial FreeGSNKE equilibrium

We start from the MAST-U-like static forward solve of example 2: the machine, an equilibrium object, a profile object and a set of coil currents. Any profile class can be used for the initial equilibrium; during the coupling the profiles are replaced by a `GeneralPprimeFFprime` object holding the $p'$ and $FF'$ received from TORAX.

In [ ]:
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,
    Zmin=-2.2, Zmax=2.2,
    nx=65, ny=129,
)

from freegsnke.jtor_update import ConstrainPaxisIp
profiles = ConstrainPaxisIp(
    eq=eq,
    paxis=8e3,    # pressure on axis [Pa]
    Ip=6e5,       # plasma current [A]
    fvac=0.5,     # fvac = R*B_tor [T m]
    alpha_m=1.8,
    alpha_n=1.2,
)

import pickle
with open("data/simple_diverted_currents_PaxisIp.pk", "rb") as f:
    currents = pickle.load(f)
for label, current in currents.items():
    eq.tokamak.set_coil_current(coil_label=label, current_value=current)

solver = GSstaticsolver.NKGSsolver(eq)
solver.solve(eq=eq, profiles=profiles, constrain=None, target_relative_tolerance=1e-8)

## TORAX configuration

The TORAX configuration is an ordinary TORAX config dictionary. Two things are specific to the coupled run:

- the `geometry` section is only used for the radial mesh (`n_rho`, `hires_factor`): the geometry itself is provided by FreeGSNKE at every step, so a lightweight `circular` placeholder is sufficient;
- the initial kinetic profiles should be smooth on axis: a profile with a cusp at $\rho = 0$ has a divergent $\mathrm{d}p/\mathrm{d}\psi$ there (since $\mathrm{d}\psi/\mathrm{d}\rho \to 0$), which the equilibrium solver cannot cope with.

Here the initial pressure is chosen to be close to that of the FreeGSNKE equilibrium above, and modest heating makes the profiles (and hence the equilibrium) evolve during the simulation.

In [ ]:
def parabolic(axis_value, edge_value, n_points=21):
    """{rho_norm: value} for a parabolic profile between axis and edge."""
    rho = np.linspace(0.0, 1.0, n_points)
    return {float(r): float(edge_value + (axis_value - edge_value) * (1 - r**2)) for r in rho}

CONFIG = {
    "profile_conditions": {
        "Ip": 6e5,
        "T_i": {0.0: parabolic(0.6, 0.1)},
        "T_e": {0.0: parabolic(0.6, 0.1)},
        "T_i_right_bc": 0.1,
        "T_e_right_bc": 0.1,
        "n_e": {0.0: parabolic(3.9e19, 1.8e19)},
        "nbar": 3e19,
        "n_e_nbar_is_fGW": False,
        "n_e_right_bc": 1e19,
        "n_e_right_bc_is_fGW": False,
        "initial_psi_mode": "geometry",   # initial poloidal flux from the FreeGSNKE equilibrium
    },
    "plasma_composition": {},
    "numerics": {
        "t_initial": 0.0,
        "t_final": 0.05,
        "fixed_dt": 0.005,      # TORAX internal time step
        "adaptive_dt": False,
        "evolve_current": True,
        "evolve_density": False,
    },
    # placeholder geometry: only n_rho is used, the geometry comes from FreeGSNKE
    "geometry": {"geometry_type": "circular", "n_rho": 25, "R_major": 0.87, "a_minor": 0.53, "B_0": 0.58},
    "neoclassical": {"bootstrap_current": {}},
    "sources": {"generic_heat": {"P_total": 2e5}, "ei_exchange": {}, "ohmic": {}},
    "transport": {"model_name": "combined", "transport_models": [{"model_name": "constant"}]},
    "solver": {"solver_type": "linear"},
    "pedestal": {},
    "time_step_calculator": {"calculator_type": "fixed"},
}
torax_config = torax.ToraxConfig.from_dict(CONFIG)

## Running the coupled simulation

`StaticEquilibriumSolver` wraps the FreeGSNKE static forward solve: at each request it reads $p'$, $FF'$ and $I_p$ from the IDS written by TORAX, optionally updates the coil currents (`coil_currents=lambda t: {...}`), solves the equilibrium (warm-started from the previous solution) and writes the result to a new IDS.

`run_loose_coupling` then advances TORAX from `t_initial` to `t_final` in intervals of `coupling_dt` (TORAX takes several of its own time steps within each interval), iterating the exchange at every interval until the relative change of the exchanged profiles (measured on the current-density scale, see `torax_coupling.profile_residual`) drops below `tolerance`. Before time stepping, the same iteration is used to make the TORAX initial state and the FreeGSNKE equilibrium consistent with each other.

The first call compiles the TORAX step function, which takes a while.

In [ ]:
equilibrium_solver = torax_coupling.StaticEquilibriumSolver(eq, profiles, target_relative_tolerance=1e-7)

result = torax_coupling.run_loose_coupling(
    torax_config,
    equilibrium_solver,
    coupling_dt=0.01,       # coupling interval [s]
    max_iterations=10,      # equilibrium/transport iterations per interval
    tolerance=1e-3,         # convergence tolerance on the exchanged p', FF'
    relaxation=0.5,         # under-relaxation of the exchanged profiles
    store_equilibria=True,  # keep a copy of the FreeGSNKE equilibrium at each coupling time
)
print("TORAX error state:", result.sim_error)
print("coupling times:", result.times)
print("iterations per coupling time:", result.iterations)
print("converged:", result.converged)

## Results

`result.torax_output` is the usual TORAX output `DataTree` (one entry per coupling time), `result.torax_history` the TORAX `StateHistory`, `result.equilibrium_ids` the FreeGSNKE equilibrium IDSs used by TORAX and `result.torax_equilibrium_ids` the TORAX IDSs handed to FreeGSNKE (containing $p'$ and $FF'$).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, residuals in enumerate(result.residuals):
    axes[0].semilogy(np.arange(1, len(residuals) + 1), residuals, "o-", label=f"t = {result.times[i]:.2f} s")
axes[0].axhline(1e-3, color="k", ls="--", lw=0.8)
axes[0].set_xlabel("coupling iteration")
axes[0].set_ylabel("relative change of exchanged p', FF'")
axes[0].legend(fontsize=8)

profiles_out = result.torax_output["profiles"]
rho = profiles_out["rho_norm"].values
for i, t in enumerate(result.times):
    axes[1].plot(rho, profiles_out["T_e"].values[i], label=f"t = {t:.2f} s")
axes[1].set_xlabel(r"$\rho_N$")
axes[1].set_ylabel("$T_e$ [keV]")
axes[1].legend(fontsize=8)
plt.tight_layout()

In [ ]:
# p' and FF' handed to FreeGSNKE (IDS convention: derivatives with respect to psi in Wb)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, ids in enumerate(result.torax_equilibrium_ids):
    p1d = ids.time_slice[0].profiles_1d
    axes[0].plot(np.asarray(p1d.psi_norm), np.asarray(p1d.dpressure_dpsi), label=f"t = {result.times[i]:.2f} s")
    axes[1].plot(np.asarray(p1d.psi_norm), np.asarray(p1d.f_df_dpsi))
axes[0].set_xlabel(r"$\psi_N$"); axes[0].set_ylabel("$p'$ [Pa/Wb]"); axes[0].legend(fontsize=8)
axes[1].set_xlabel(r"$\psi_N$"); axes[1].set_ylabel("$FF'$ [T$^2$ m$^2$/Wb]")
plt.tight_layout()

In [ ]:
# equilibrium at the start and at the end of the coupled simulation
fig, axes = plt.subplots(1, 2, figsize=(8, 8))
for ax, index in zip(axes, [0, -1]):
    equilibrium = result.equilibria[index]
    equilibrium.plot(axis=ax, show=False)
    equilibrium.tokamak.plot(axis=ax, show=False)
    ax.set_xlim(0.1, 2.15)
    ax.set_ylim(-2.25, 2.25)
    ax.set_title(f"t = {result.times[index]:.2f} s, $R_{{mag}}$ = {equilibrium.Rmagnetic():.3f} m")
plt.tight_layout()

The FreeGSNKE IDSs can be saved to netCDF with `freegsnke.imas_read_write.save_equilibrium_ids` for use in other IMAS-aware tools.